# Rescore multilingual + order-ops with fixed logic

Uses the fitted lens from the prior kernel (attached as a kernel data source, so it appears at /kaggle/input/jacobian-lens-qwen-replication/).

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/anthropics/jacobian-lens.git',
    '--upgrade', 'transformers>=5.5'])
import torch
print(torch.cuda.get_device_name(0), '| torch:', torch.__version__)

In [ ]:
import os, subprocess
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import torch, transformers, jlens, json, pathlib, glob

# Find the .pt lens file (path may vary depending on how kernel is attached)
cands = glob.glob('/kaggle/input/**/*jlens.pt', recursive=True)
print('Candidate lens files:', cands)
assert cands, 'No lens file found'
LENS_PATH = cands[0]

MODEL = 'Qwen/Qwen2.5-3B-Instruct'
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, attn_implementation='sdpa', low_cpu_mem_usage=True
).to('cuda')
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL)
model = jlens.from_hf(hf_model, tokenizer, compile=True)
lens = jlens.JacobianLens.load(LENS_PATH)
print('n_layers:', model.n_layers, ' lens.n_prompts:', lens.n_prompts)

In [ ]:
!git clone -q --depth=1 https://github.com/anthropics/jacobian-lens.git /tmp/jl
EVALS_DIR = pathlib.Path('/tmp/jl/data/evaluations')
K_VALUES = (1, 5, 10)

def tokens_of(word):
    ids = []
    for v in {word, ' '+word, word.lower(), ' '+word.lower()}:
        e = tokenizer(v, add_special_tokens=False).input_ids
        if len(e) == 1:
            ids.append(e[0])
    return sorted(set(ids))

def min_rank_across_layers(lens_logits, token_ids):
    if not token_ids: return 10**9
    best = 10**9
    for _, logits in lens_logits.items():
        order = logits[0].argsort(descending=True)
        rank_lookup = torch.empty_like(order)
        rank_lookup[order] = torch.arange(order.numel(), device=order.device)
        cand = rank_lookup[torch.tensor(token_ids, device=order.device)]
        r = int(cand.min().item()) + 1
        if r < best: best = r
    return best

def score_eval(path, mode):
    data = json.load(path.open())
    items = data['items']
    per_item = {k: [] for k in K_VALUES}
    n_dropped = 0
    for item in items:
        try:
            lens_logits, _, _ = lens.apply(model, item['prompt'], positions=[-1])
        except Exception:
            continue
        inters = item['intermediates']
        if not isinstance(inters, list):
            inters = [inters]
        targets = [inters[0]] if mode == 'original' else inters
        hits = {k: 0 for k in K_VALUES}
        total = 0
        for t in targets:
            if not isinstance(t, str): t = str(t)
            tok_ids = tokens_of(t)
            if not tok_ids:
                n_dropped += 1
                continue
            r = min_rank_across_layers(lens_logits, tok_ids)
            for k in K_VALUES:
                if r <= k: hits[k] += 1
            total += 1
        if total:
            for k in K_VALUES:
                per_item[k].append(hits[k]/total)
    return {
        'n_items_scored': len(per_item[1]),
        'n_multitoken_dropped': n_dropped,
        'pass_at_k': {f'pass@{k}': sum(per_item[k])/max(1,len(per_item[k])) for k in K_VALUES}
    }

results = {}
for eval_name in ['lens-eval-multilingual', 'lens-eval-order-ops']:
    path = EVALS_DIR / f'{eval_name}.json'
    print(f'\n=== {eval_name} ===')
    for mode in ('original', 'all_intermediates'):
        r = score_eval(path, mode)
        results.setdefault(eval_name, {})[mode] = r
        print(f'  [{mode}]  n={r["n_items_scored"]}  dropped_multitok={r["n_multitoken_dropped"]}  {r["pass_at_k"]}')

pathlib.Path('/kaggle/working/rescore.json').write_text(json.dumps(results, indent=2))
print('\nSaved /kaggle/working/rescore.json')